# Evaluation - Object Detection

`advsecurenet` supports evaluation for object detection. Currently supported evaluations are: 

1. **mean Average Precision**: The metric calculating the mean of the Average Precision computed per class. AP summarizes the precision-recall tradeoff per class.

There are two ways to evaluate the model using `advsecurenet` API. We can either utilize the `ODAttacker` or we can manually iterate over the dataset and calculate the evaluation metrics.

**Note:** `advsecurenet` CLI also supports evaluation.

## Adversarial Evaluation

### Using the ODAttacker

In [ ]:
from advsecurenet.shared.types.configs.attack_configs.attacker_config import (
    AttackerConfig,
)
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig


In [ ]:
# Define the model
architecture = {
    "num_classes": 80,  # Number of classes in the dataset (e.g., COCO has 80 classes)
}
model = ModelFactory.create_model(
    model_name="CustomYolov5Model", 
    pretrained=False, 
    is_external=True,
    model_weights_path="../../../model_weights/yolov5s.pt",
    model_arch_path="../../../advsecurenet/models/CustomODModels/CustomYolov5Model.py",
    architecture=architecture,
)

In [ ]:
# Lets define the preprocessing configuration we want to use
clip_max = 255.0
preprocess_config = PreprocessConfig(steps=[
    PreprocessStep(name="Resize", params={"size": (640, 640)}),
    # Turn a H×W×C numpy array (0–255) into a FloatTensor C×H×W in [0,1]
    PreprocessStep(name="ToTensor"),
    # Cast dtype to float32
    PreprocessStep(name="ToDtype", params={"dtype": "torch.float32"}),
])

# Define the dataset
dataset = DatasetFactory.load_dataset(
    dataset_name="COCO", preprocessing=preprocess_config)
train_data = dataset['train']
test_data = dataset['test']

In [ ]:
from torch.utils.data import Subset
if 'test_data' in locals() and hasattr(test_data, '__len__') and hasattr(test_data, '__getitem__'):
    original_len = len(test_data)
    num_samples_to_keep = 20
    if original_len == 0:
        print("test_data is empty. Cannot create a subset.")
    elif original_len < num_samples_to_keep:
        print(f"test_data has only {original_len} sample(s), which is less than the desired {num_samples_to_keep}. Using all available {original_len} sample(s).")
    else:
        subset_indices = list(range(num_samples_to_keep))
        test_data = Subset(test_data, subset_indices)
        print(f"Reduced test_data from {original_len} to {len(test_data)} samples.")
else:
    print("Warning: 'test_data' for COCO dataset not found or is not a valid dataset. Please ensure it's loaded correctly in a previous cell.")


In [ ]:
# Define the dataloder
dataloader = DataLoaderFactory.create_od_dataloader(dataset=test_data, batch_size=4)

In [ ]:
from advsecurenet.computer_vision.object_detection.attacks.adversarial_patch_based.dpatch import DPatchAttackConfig, DPatch
from advsecurenet.models.detector_factory import get_object_detector
# Define the device config
device_name = "cuda:0" # change if necessary
device = DeviceConfig(processor=device_name)
# Define object detector config
object_detector_config = {
    "input_shape": (3, 640, 640),
    "device_type": device_name,
    "clip_values": (0.0, clip_max),
    "conf_thresh": 0.25,
}
detector = get_object_detector(config=object_detector_config, existing_model=model.model)

# Define the DPatch config
config = DPatchAttackConfig(
    object_detector=detector,
    patch_shape=(3, 200, 200),
    learning_rate=1.99,
    max_iter=800,
    target_label=None,  # None for untargeted attack
    device=device,
)
attack = DPatch(config)

In [ ]:
from advsecurenet.computer_vision.object_detection.attacks.attacker import AdversarialPatchODAttacker
from advsecurenet.shared.types.configs.attack_configs.od_attacker_config import (
    ODAttackerConfig,
)

evaluators = [
    "mean_average_precision",
]
attacker_config = ODAttackerConfig(
    model=model,
    attack=attack,
    dataloader=dataloader,
    device=device,
    return_adversarial_images=False,
    evaluators=evaluators,
)

attacker = AdversarialPatchODAttacker(config=attacker_config)

attacker.execute()

### Using the Evaluator

If you don't want to use the `ODAttacker` class, you can manually iterate over the dataset and calculate the evaluation metrics. Here is an example of how to do that:


In [ ]:
from advsecurenet.evaluation.od_adversarial_evaluator import (
    ObjectDetectorAdversarialEvaluator,
)

In [ ]:
with ObjectDetectorAdversarialEvaluator(evaluators=["mean_average_precision"]) as evaluator:
    for data_batch in dataloader:
        images_preprocessed, targets, images = attacker.process_batch(data_batch)
        adv_images = attack.apply_patch(images_preprocessed)
        evaluator.update(
            model=detector.inference_model,
            original_images=images,
            adversarial_images=adv_images,
            targets=targets,
        )

print(evaluator.get_results())